In [ ]:
# imports
import pandas as pd
import matplotlib.pyplot as plt

from finvizfinance.quote import finvizfinance

import os

In [217]:
# directory path
dir_path = r"..\14_stocks_analysis\00_data"
if not os.path.isdir(dir_path):
    raise FileNotFoundError(f'Directory does not exist: {dir_path}')
print(f'Using data directory: {dir_path}')

Using data directory: ..\14_stocks_analysis\00_data


In [218]:
# concatenate all tabular files into a single dataframe
files = [
    file for file in os.listdir(dir_path)
    if file.lower().endswith(('.csv', '.xlsx', '.xls'))
]
if not files:
    raise ValueError(f'No CSV/XLSX/XLS files found in: {dir_path}')

def load_table(file_name):
    file_path = os.path.join(dir_path, file_name)
    if file_name.lower().endswith('.csv'):
        return pd.read_csv(file_path)
    return pd.read_excel(file_path)

# file name have format like 2603, 2602..first two digits are year and last two are month
# when concatenating, we need to add a column with the date extracted from the file name
def extract_date(file_name):
    year = int(file_name[:2]) + 2000
    month = int(file_name[2:4])
    return pd.Timestamp(year=year, month=month, day=1)

df = pd.concat([load_table(file).assign(date=extract_date(file)) for file in sorted(files)], ignore_index=True)

# lowercase column names
df.columns = df.columns.str.lower()

df.head()

,unnamed: 0,symbol,stock,market cap,price,fair value (%),z-score,f-score,m-score,value generation,date
0,1.0,Lululemon Athletica Inc.,Lululemon Athletica Inc.,20.11B,168.18,116.31,7.43,8.0,-2.78,Robust,2025-11-01
1,NaN,LULU,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-11-01
2,2.0,"PayPal Holdings, Inc.","PayPal Holdings, Inc.",58.63B,60.57,100.92,1.96,7.0,-2.99,Robust,2025-11-01
3,NaN,PYPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-11-01
4,3.0,Adobe Inc.,Adobe Inc.,139.08B,324.19,82.13,8.92,7.0,-2.49,Robust,2025-11-01


In [219]:
# create a copy of symbol column named 'symbol_copy'
df['symbol_copy'] = df['symbol']

In [220]:
# the series in df.symbol_copy, should be shifted backwards by 1 row, to align with the stock name in the same row
df['symbol_copy'] = df['symbol_copy'].shift(-1)

In [221]:
# drop all rows where column 'unnamed: 0' is null
df = df.dropna(subset=['unnamed: 0'])

In [222]:
# drop columns 'unnamed: 0', 'symbol'
df = df.drop(columns=['unnamed: 0', 'symbol'])

# rename column 'symbol_copy' to 'symbol'
df = df.rename(columns={'symbol_copy': 'symbol'})

# columns order 'date', 'symbol', 'stock', 'market cap', 'price', 'fair value (%)', 'z-score', 'f-score', 'm-score', 'value generation'
df = df[['date', 'symbol', 'stock', 'market cap', 'price', 'fair value (%)', 'z-score', 'f-score', 'm-score', 'value generation']]

In [223]:
df.columns

Index(['date', 'symbol', 'stock', 'market cap', 'price', 'fair value (%)',
       'z-score', 'f-score', 'm-score', 'value generation'],
      dtype='object')

In [224]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 250 entries, 0 to 498
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              250 non-null    datetime64[us]
 1   symbol            250 non-null    object        
 2   stock             250 non-null    object        
 3   market cap        250 non-null    object        
 4   price             250 non-null    float64       
 5   fair value (%)    250 non-null    float64       
 6   z-score           250 non-null    float64       
 7   f-score           250 non-null    object        
 8   m-score           250 non-null    float64       
 9   value generation  250 non-null    object        
dtypes: datetime64[us](1), float64(4), object(5)
memory usage: 21.5+ KB


In [225]:
# create a copy of the dataframe
df_copy = df.copy()

# keep only date, symbol columns
df_copy = df_copy[['date', 'symbol']]

# group by symbol and count the number of occurrences of each symbol
symbol_counts = df_copy.groupby('symbol').size().reset_index(name='count')

# sort the dataframe by count in descending order
symbol_counts = symbol_counts.sort_values(by='count', ascending=False)

# display the top 10 symbols with the most occurrences
print(symbol_counts.head(10))

   symbol  count
2    ADBE      5
18    CLX      5
41   FTNT      5
93   PYPL      5
20    CMG      4
7     AVY      4
17     CL      4
35     EW      4
50    HUM      4
25   DECK      4


In [226]:
# drop duplicates in df.symbol, keep last occurrence
df = df.drop_duplicates(subset='symbol', keep='last')

# merge df with symbol_counts on symbol, keeping only rows that are in df
df = df.merge(symbol_counts, on='symbol', how='inner')

# sort df by count in descending order
df = df.sort_values(by='count', ascending=False)

df.head(10)

,date,symbol,stock,market cap,price,fair value (%),z-score,f-score,m-score,value generation,count
83,2026-03-01,PYPL,"PayPal Holdings, Inc.",42.66B,444859.00,87.65,1.99,8.0,-2.54,Robust,5
72,2026-03-01,CLX,The Clorox Company,12.44B,102.28,87.39,2.44,6.0,-3.16,Robust,5
69,2026-03-01,ADBE,Adobe Inc.,98.74B,2408252.00,101.55,7.53,7.0,-3.41,Robust,5
117,2026-03-01,FTNT,"Fortinet, Inc.",59.24B,79725.00,17.65,5.22,7.0,-2.23,Robust,5
39,2026-02-01,CMG,"Chipotle Mexican Grill, Inc.",48.27B,36.71,50.78,7.74,8.0,-1.70,Robust,4
42,2026-02-01,AVY,Avery Dennison Corporation,14.49B,187.20,29.80,3.93,9.0,-2.89,Robust,4
41,2026-02-01,DECK,Deckers Outdoor Corporation,15.98B,108.74,33.70,14.21,9.0,-2.89,Robust,4
55,2026-02-01,TMO,Thermo Fisher Scientific Inc.,192.77B,512.69,12.11,4.07,8.0,-2.65,Robust,4
66,2026-02-01,JKHY,"Jack Henry & Associates, Inc.",12.20B,168.43,23.66,10.78,8.0,-2.31,Resilient,4
74,2026-03-01,LDOS,"Leidos Holdings, Inc.",20.35B,153.66,78.00,4.15,8.0,-2.51,Robust,4


In [227]:
# market cap column is object. We need to convert it to numeric, removing any non-numeric characters ("T" indicating trillions, 
# "B", indicating billions, "M" indicating millions)
# These characters are at the end of the string, so we can use str.replace to remove them and then convert to numeric
# They will be added in a new column named 'market cap um'

# take last character of market cap column and add a new column 'market cap um' with the value of the last character
df['market cap um'] = df['market cap'].str[-1]

# replace 'T', 'B', 'M' with "" in market cap column and convert to numeric
df['market cap'] = df['market cap'].str.replace(r'[TBM]', '', regex=True).astype(float)


In [228]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 119 entries, 83 to 118
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              119 non-null    datetime64[us]
 1   symbol            119 non-null    object        
 2   stock             119 non-null    object        
 3   market cap        119 non-null    float64       
 4   price             119 non-null    float64       
 5   fair value (%)    119 non-null    float64       
 6   z-score           119 non-null    float64       
 7   f-score           119 non-null    object        
 8   m-score           119 non-null    float64       
 9   value generation  119 non-null    object        
 10  count             119 non-null    int64         
 11  market cap um     119 non-null    object        
dtypes: datetime64[us](1), float64(5), int64(1), object(5)
memory usage: 12.1+ KB


In [229]:
# if market cap um is M, market cap / 1000, if market cap um is T, market cap * 1000, if market cap um is B, market cap / 1
def convert_market_cap(row):
    if row['market cap um'] == 'M':
        return row['market cap'] / 1000
    elif row['market cap um'] == 'T':
        return row['market cap'] * 1000
    elif row['market cap um'] == 'B':
        return row['market cap']
    else:
        return row['market cap']

# apply the function to the dataframe and create a new column 'market cap converted'
df['market cap converted'] = df.apply(convert_market_cap, axis=1)

In [230]:
# drop market cap	
df = df.drop(columns=['market cap'])

# rename market cap converted to market cap
df = df.rename(columns={'market cap converted': 'market cap'})

# columns order 'date', 'symbol', 'stock', 'price', 'fair value (%)', 'z-score', 'f-score', 'm-score', 'value generation', 'market cap um', 'market cap'
df = df[['date', 'symbol', 'stock', 'price', 'fair value (%)', 'z-score', 
'f-score', 'm-score', 'value generation', 'market cap um', 'market cap', 'count']]


In [231]:
# market cap filter
# drop all rows where market cap um is M
df = df[df['market cap um'] != 'M']

# drop all rows where market cap um is B and market cap is less than 10
df = df[~((df['market cap um'] == 'B') & (df['market cap'] < 10))]

In [234]:
# print length of dataframe
print("Number of rows in the dataframe:", len(df))

# print lenght of unique symbols in the dataframe (for comparison with the original dataframe)
print("Number of unique symbols in the dataframe:", df['symbol'].nunique())

Number of rows in the dataframe: 108
Number of unique symbols in the dataframe: 108


In [232]:
df.sample(10)

,date,symbol,stock,price,fair value (%),z-score,f-score,m-score,value generation,market cap um,market cap,count
27,2026-01-01,NWS,News Corporation,27.39,90.14,2.36,7.0,-1.94,Robust,B,15.42,2
25,2026-01-01,VRSK,"Verisk Analytics, Inc.",184.68,55.18,8.36,8.0,-3.25,Robust,B,25.76,3
104,2026-03-01,EW,Edwards Lifesciences Corporation,79.62,4.33,11.63,6.0,-2.72,Robust,B,46.56,4
22,2025-12-01,META,"Meta Platforms, Inc.",663.29,0.28,12.78,9.0,-3.03,Robust,T,1670.00,2
109,2026-03-01,EA,Electronic Arts Inc.,202.62,-16.56,5.72,5.0,-3.04,Robust,B,50.66,1
110,2026-03-01,ROK,"Rockwell Automation, Inc.",349.12,28.42,9.14,9.0,-2.95,Resilient,B,39.21,2
108,2026-03-01,KR,The Kroger Co.,73965.00,-11.99,3.55,5.0,-3.37,Robust,B,48.23,2
55,2026-02-01,TMO,Thermo Fisher Scientific Inc.,512.69,12.11,4.07,8.0,-2.65,Robust,B,192.77,4
68,2026-02-01,RMD,ResMed Inc.,259.05,20.44,12.94,8.0,-2.56,Resilient,B,37.94,1
6,2025-11-01,GD,General Dynamics Corporation,340.34,8.80,4.01,6.0,-3.58,Robust,B,91.48,1


In [235]:
# list of tickers in the dataframe
tickers = df['symbol'].tolist()

# TODO: change it later, we will keep first 3 tickers for testing
tickers = tickers[:3]

In [236]:
tickers

['PYPL', 'CLX', 'ADBE']

In [ ]:
# # Examples from finvizfinance documentation
# stock = finvizfinance('tsla')

# stock.TickerCharts(out_dir='asset')

# stock_fundament = stock.ticker_fundament()
# stock_description = stock.ticker_description()
# outer_ratings_df = stock.ticker_outer_ratings()
# news_df = stock.ticker_news()
# inside_trader_df = stock.ticker_inside_trader()